# 04 · Distributed ml pipeline

Simulate five nodes with LightGBM models: engine-ID block and operational k-means partitions, majority, weighted and confidence-vote aggregation.

- **Reads:** feature parquet files, notebook 03 outputs
- **Writes:** `distributed_results.csv`, `deployment_metrics.csv`, `node_metadata.csv`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
import pandas as pd

df = pd.read_parquet(
"/kaggle/input/datasets/<kaggle-user>/nb-3-again-outputs/FD001_train_features.parquet")

print([
    c for c in df.columns
    if "label" in c.lower() or "failure" in c.lower()
])

['failure_within_horizon']


In [2]:
!pip install -q lightgbm codecarbon

import os
import gc
import json
import time
import shutil
import warnings
import tracemalloc
import tempfile

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from codecarbon import EmissionsTracker

from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

print("✅ NB04 Cell 1 completed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 62.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.2.0 requires cryptography<47,>=42, but you have cryptography 49.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 49.0.0 which is inc

In [3]:
WORKING_DIR = Path("/kaggle/working")

DATA_DIR = Path("/kaggle/input/datasets/<kaggle-user>")

RAW_DATA_DIR = DATA_DIR / "nasa-cmaps"

NB01_DIR = DATA_DIR / "nb-1-again-outputs"

NB03_DIR = (
    DATA_DIR / "nb-03-again-outputs"
    if (DATA_DIR / "nb-03-again-outputs").exists()
    else DATA_DIR / "nb-3-again-outputs"
)

OUTPUT_ROOT = WORKING_DIR / "NB-4-AGAIN-outputs"

MODELS_DIR = OUTPUT_ROOT / "models"
PRED_DIR = OUTPUT_ROOT / "predictions"
PARTITIONS_DIR = OUTPUT_ROOT / "partitions"
RESULTS_DIR = OUTPUT_ROOT / "results"
META_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [
    MODELS_DIR,
    PRED_DIR,
    PARTITIONS_DIR,
    RESULTS_DIR,
    META_DIR,
    LOG_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("NB03_DIR:", NB03_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

print("✅ NB04 Cell 2 completed")

NB03_DIR: /kaggle/input/datasets/<kaggle-user>/nb-03-again-outputs
OUTPUT_ROOT: /kaggle/working/NB-4-AGAIN-outputs
✅ NB04 Cell 2 completed


In [4]:
DATASETS = ["FD001", "FD002", "FD003", "FD004"]

SEEDS = [42, 52, 62, 72, 82]

TARGET_COL = "failure_within_horizon"

PARTITION_METHODS = [
    "engine_id_block",
    "operational_clustering"
]

AGGREGATIONS = [
    "majority_vote",
    "weighted_vote",
    "confidence_vote"
]

NODE_COUNT = 5

INDEX_COLS = [
    "dataset",
    "engine_id",
    "cycle",
    "split"
]

LABEL_COLS = [
    "RUL",
    "rul",
    "label_w10",
    "label_w20",
    "label_w30",
    "failure_within_horizon"
]

run_config = {
    "datasets": DATASETS,
    "seeds": SEEDS,
    "target": TARGET_COL,
    "node_count": NODE_COUNT,
    "partition_methods": PARTITION_METHODS,
    "aggregations": AGGREGATIONS
}

with open(META_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

print("✅ NB04 Cell 3 completed")

✅ NB04 Cell 3 completed


In [5]:
# ============================================================
# NB04 Cell 4: Dataset Loader
# ============================================================

NB02_DIR = DATA_DIR / "/kaggle/input/datasets/<kaggle-user>/nb-3-again-outputs"

assert NB02_DIR.exists(), f"Missing: {NB02_DIR}"

def load_dataset(dataset_name):

    train_path = NB02_DIR / f"{dataset_name}_train_features.parquet"
    val_a_path = NB02_DIR / f"{dataset_name}_val_a_features.parquet"
    val_b_path = NB02_DIR / f"{dataset_name}_val_b_features.parquet"
    test_path  = NB02_DIR / f"{dataset_name}_test_features.parquet"

    for path in [train_path, val_a_path, val_b_path, test_path]:
        assert path.exists(), f"Missing file: {path}"

    train_df = pd.read_parquet(train_path)
    val_a_df = pd.read_parquet(val_a_path)
    val_b_df = pd.read_parquet(val_b_path)
    test_df = pd.read_parquet(test_path)

    train_df = train_df.loc[:, ~train_df.columns.duplicated()]
    val_a_df = val_a_df.loc[:, ~val_a_df.columns.duplicated()]
    val_b_df = val_b_df.loc[:, ~val_b_df.columns.duplicated()]
    test_df = test_df.loc[:, ~test_df.columns.duplicated()]

    feature_cols = [
        c for c in train_df.columns
        if c not in INDEX_COLS + LABEL_COLS
    ]

    assert len(feature_cols) == 342, (
        f"Expected 342 features, got {len(feature_cols)}"
    )

    assert TARGET_COL in train_df.columns

    return (
        train_df,
        val_a_df,
        val_b_df,
        test_df,
        feature_cols
    )

print("✅ NB04 Cell 4 completed")

✅ NB04 Cell 4 completed


In [6]:
def engine_id_block_partition(df):

    unique_engines = sorted(df["engine_id"].unique())

    partitions = {}

    splits = np.array_split(unique_engines, NODE_COUNT)

    for idx, engines in enumerate(splits):
        partitions[idx] = df[
            df["engine_id"].isin(engines)
        ].copy()

    return partitions


def operational_clustering_partition(df, feature_cols):

    early = df[df["cycle"] <= 20]

    agg = (
        early.groupby("engine_id")[feature_cols]
        .mean()
        .reset_index()
    )

    kmeans = KMeans(
        n_clusters=NODE_COUNT,
        random_state=42,
        n_init=10
    )

    agg["cluster"] = kmeans.fit_predict(
        agg[feature_cols]
    )

    mapping = dict(
        zip(agg["engine_id"], agg["cluster"])
    )

    df["cluster"] = df["engine_id"].map(mapping)

    partitions = {}

    for cluster in range(NODE_COUNT):
        partitions[cluster] = df[
            df["cluster"] == cluster
        ].copy()

    return partitions

print("✅ NB04 Cell 5 completed")

✅ NB04 Cell 5 completed


In [7]:
def parameter_count(model):
    return model.booster_.num_trees()


def estimate_flops(model, n_features):
    return parameter_count(model) * n_features * 2


def get_model_size_mb(model):

    with tempfile.NamedTemporaryFile(
        suffix=".joblib"
    ) as tmp:

        joblib.dump(model, tmp.name)

        return os.path.getsize(tmp.name) / (1024**2)


def optimize_threshold(y_true, probs):

    best_t = 0.5
    best_f1 = -1

    for t in np.arange(0.05, 0.96, 0.05):

        preds = (probs >= t).astype(int)

        score = f1_score(
            y_true,
            preds,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_t = t

    return best_t


print("✅ NB04 Cell 6 completed")

✅ NB04 Cell 6 completed


In [8]:
def create_model(seed):

    return LGBMClassifier(
        n_estimators=200,
        num_leaves=31,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    )

print("✅ NB04 Cell 7 completed")

✅ NB04 Cell 7 completed


In [9]:
def aggregate_predictions(
    probs,
    method,
    weights=None
):

    probs = np.array(probs)

    if method == "confidence_vote":
        return probs.mean(axis=0)

    binary = (probs >= 0.5).astype(int)

    if method == "majority_vote":
        return binary.mean(axis=0)

    if method == "weighted_vote":
        return np.average(
            binary,
            axis=0,
            weights=weights
        )

print("✅ NB04 Cell 8 completed")

✅ NB04 Cell 8 completed


In [10]:
train_df, val_a_df, val_b_df, test_df, FEATURE_COLS = load_dataset("FD001")

print(f"Train : {train_df.shape}")
print(f"Val A : {val_a_df.shape}")
print(f"Val B : {val_b_df.shape}")
print(f"Test  : {test_df.shape}")

print(f"Features: {len(FEATURE_COLS)}")

assert len(FEATURE_COLS) == 342
assert TARGET_COL in train_df.columns

print("✅ NB04 dataset loader validated")

Train : (14507, 346)
Val A : (1759, 346)
Val B : (1007, 346)
Test  : (3358, 346)
Features: 342
✅ NB04 dataset loader validated


In [11]:
# ============================================================
# NB04 Cell 9A: Create Partitions
# ============================================================

def create_partitions(df, method, feature_cols):

    if method == "engine_id_block":
        return engine_id_block_partition(df)

    elif method == "operational_clustering":
        return operational_clustering_partition(
            df.copy(),
            feature_cols
        )

    raise ValueError(f"Unknown partition method: {method}")


print("✅ NB04 Cell 9A completed")

✅ NB04 Cell 9A completed


In [12]:
# ============================================================
# NB04 Cell 9B: Distributed Experiment
# ============================================================

distributed_results = []
deployment_results = []
node_metadata = []

def run_distributed_experiment(
    dataset_name,
    partition_method,
    aggregation_method,
    seed
):

    gc.collect()

    (
        train_df,
        val_a_df,
        val_b_df,
        test_df,
        feature_cols
    ) = load_dataset(dataset_name)

    train_parts = create_partitions(
        train_df,
        partition_method,
        feature_cols
    )

    val_a_parts = create_partitions(
        val_a_df,
        partition_method,
        feature_cols
    )

    node_models = []
    node_weights = []

    total_train_time = 0
    total_train_memory = 0
    total_energy = 0

    for node_id in range(NODE_COUNT):

        node_train = train_parts[node_id]
        node_val = val_a_parts[node_id]

        X_train = node_train[feature_cols]
        y_train = node_train[TARGET_COL]

        X_val = node_val[feature_cols]
        y_val = node_val[TARGET_COL]

        model = create_model(seed)

        tracker = EmissionsTracker(
            save_to_file=False,
            log_level="error",
            allow_multiple_runs=True
        )

        tracemalloc.start()

        tracker.start()
        start = time.time()

        model.fit(X_train, y_train)

        train_time = time.time() - start

        energy = tracker.stop() or 0.0

        _, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        total_train_time += train_time
        total_train_memory += peak_mem / (1024**2)
        total_energy += energy

        val_probs = model.predict_proba(X_val)[:, 1]

        val_preds = (val_probs >= 0.5).astype(int)

        node_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        node_models.append(model)
        node_weights.append(max(node_f1, 1e-6))

        node_metadata.append({
            "dataset": dataset_name,
            "partition_method": partition_method,
            "aggregation_method": aggregation_method,
            "seed": seed,
            "node_id": node_id,
            "samples": len(node_train),
            "validation_f1": node_f1
        })

    X_val_b = val_b_df[feature_cols]
    y_val_b = val_b_df[TARGET_COL]

    val_probs = []

    for model in node_models:
        val_probs.append(
            model.predict_proba(X_val_b)[:, 1]
        )

    agg_val_probs = aggregate_predictions(
        val_probs,
        aggregation_method,
        node_weights
    )

    threshold = optimize_threshold(
        y_val_b,
        agg_val_probs
    )

    X_test = test_df[feature_cols]
    y_test = test_df[TARGET_COL]

    tracemalloc.start()

    infer_start = time.time()

    test_probs = []

    for model in node_models:
        test_probs.append(
            model.predict_proba(X_test)[:, 1]
        )

    agg_test_probs = aggregate_predictions(
        test_probs,
        aggregation_method,
        node_weights
    )

    inference_time = time.time() - infer_start

    _, peak_infer = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    preds = (agg_test_probs >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(
            y_test,
            preds,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            preds,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            preds,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test,
            agg_test_probs
        )
    }

    latency_ms = (
        inference_time / len(X_test)
    ) * 1000

    throughput_sps = (
        len(X_test) / inference_time
    )

    model_size_mb = sum(
        get_model_size_mb(m)
        for m in node_models
    )

    parameter_cnt = sum(
        parameter_count(m)
        for m in node_models
    )

    flops = sum(
        estimate_flops(m, len(feature_cols))
        for m in node_models
    )

    return {
        "dataset": dataset_name,
        "partition_method": partition_method,
        "aggregation_method": aggregation_method,
        "seed": seed,
        "threshold": threshold,
        **metrics,
        "train_time_sec": total_train_time,
        "inference_time_sec": inference_time,
        "latency_ms": latency_ms,
        "throughput_sps": throughput_sps,
        "memory_mb_training": total_train_memory,
        "memory_mb_inference": peak_infer / (1024**2),
        "model_size_mb": model_size_mb,
        "parameter_count": parameter_cnt,
        "flops_estimated": flops,
        "energy_kwh": total_energy,
        "models": node_models,
        "probabilities": agg_test_probs,
        "predictions": preds,
        "test_df": test_df
    }

print("✅ NB04 Cell 9B completed")

✅ NB04 Cell 9B completed


In [13]:
# ============================================================
# NB04 Cell 10: Execute Distributed Runs
# ============================================================

total_runs = (
    len(DATASETS)
    * len(PARTITION_METHODS)
    * len(AGGREGATIONS)
    * len(SEEDS)
)

current = 0

for dataset_name in DATASETS:

    for partition_method in PARTITION_METHODS:

        for aggregation_method in AGGREGATIONS:

            for seed in SEEDS:

                current += 1

                print(
                    f"\n[{current}/{total_runs}] "
                    f"{dataset_name} | "
                    f"{partition_method} | "
                    f"{aggregation_method} | "
                    f"seed={seed}"
                )

                try:

                    result = run_distributed_experiment(
                        dataset_name,
                        partition_method,
                        aggregation_method,
                        seed
                    )

                    models = result.pop("models")
                    probs = result.pop("probabilities")
                    preds = result.pop("predictions")
                    test_df = result.pop("test_df")

                    model_dir = (
                        MODELS_DIR /
                        "distributed" /
                        dataset_name /
                        partition_method /
                        aggregation_method /
                        f"seed_{seed}"
                    )

                    model_dir.mkdir(
                        parents=True,
                        exist_ok=True
                    )

                    for idx, model in enumerate(models):

                        joblib.dump(
                            model,
                            model_dir / f"node_{idx}.joblib"
                        )

                    pred_df = pd.DataFrame({
                        "engine_id": test_df["engine_id"],
                        "cycle": test_df["cycle"],
                        "y_true": test_df[TARGET_COL],
                        "probability": probs,
                        "prediction": preds
                    })

                    pred_path = (
                        PRED_DIR /
                        f"{dataset_name}_{partition_method}_{aggregation_method}_seed{seed}.csv"
                    )

                    pred_df.to_csv(
                        pred_path,
                        index=False
                    )

                    deployment_results.append({
                        k: result[k]
                        for k in [
                            "dataset",
                            "partition_method",
                            "aggregation_method",
                            "seed",
                            "memory_mb_training",
                            "memory_mb_inference",
                            "latency_ms",
                            "throughput_sps",
                            "model_size_mb",
                            "parameter_count",
                            "flops_estimated",
                            "energy_kwh"
                        ]
                    })

                    distributed_results.append(result)

                    print(
                        f"✅ F1={result['f1']:.4f} | "
                        f"AUC={result['roc_auc']:.4f}"
                    )

                except Exception as e:

                    print(f"❌ {e}")

print("✅ NB04 Cell 10 completed")


[1/120] FD001 | engine_id_block | majority_vote | seed=42


[codecarbon WARNING @ 22:40:52] Multiple instances of codecarbon are allowed to run at the same time.


✅ F1=0.8681 | AUC=0.9853

[2/120] FD001 | engine_id_block | majority_vote | seed=52
✅ F1=0.8681 | AUC=0.9853

[3/120] FD001 | engine_id_block | majority_vote | seed=62
✅ F1=0.8681 | AUC=0.9853

[4/120] FD001 | engine_id_block | majority_vote | seed=72
✅ F1=0.8681 | AUC=0.9853

[5/120] FD001 | engine_id_block | majority_vote | seed=82
✅ F1=0.8681 | AUC=0.9853

[6/120] FD001 | engine_id_block | weighted_vote | seed=42
✅ F1=0.8681 | AUC=0.9853

[7/120] FD001 | engine_id_block | weighted_vote | seed=52
✅ F1=0.8681 | AUC=0.9853

[8/120] FD001 | engine_id_block | weighted_vote | seed=62
✅ F1=0.8681 | AUC=0.9853

[9/120] FD001 | engine_id_block | weighted_vote | seed=72
✅ F1=0.8681 | AUC=0.9853

[10/120] FD001 | engine_id_block | weighted_vote | seed=82
✅ F1=0.8681 | AUC=0.9853

[11/120] FD001 | engine_id_block | confidence_vote | seed=42
✅ F1=0.8850 | AUC=0.9964

[12/120] FD001 | engine_id_block | confidence_vote | seed=52
✅ F1=0.8850 | AUC=0.9964

[13/120] FD001 | engine_id_block | confiden

In [14]:
# ============================================================
# NB04 Cell 11: Save Results
# ============================================================

results_df = pd.DataFrame(distributed_results)

deployment_df = pd.DataFrame(deployment_results)

node_df = pd.DataFrame(node_metadata)

results_df.to_csv(
    RESULTS_DIR / "distributed_results.csv",
    index=False
)

deployment_df.to_csv(
    RESULTS_DIR / "deployment_metrics.csv",
    index=False
)

node_df.to_csv(
    META_DIR / "node_metadata.csv",
    index=False
)

print(results_df.shape)
print(deployment_df.shape)
print(node_df.shape)

print("✅ NB04 Cell 11 completed")

(120, 20)
(120, 12)
(600, 7)
✅ NB04 Cell 11 completed


In [15]:
# ============================================================
# NB04 Cell 12: Validation
# ============================================================

required_cols = [
    "dataset",
    "partition_method",
    "aggregation_method",
    "seed",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "train_time_sec",
    "inference_time_sec",
    "latency_ms",
    "throughput_sps",
    "memory_mb_training",
    "memory_mb_inference",
    "model_size_mb",
    "parameter_count",
    "flops_estimated",
    "energy_kwh"
]

assert len(results_df) == 120

assert not results_df[
    required_cols
].isnull().any().any()

assert (results_df["throughput_sps"] > 0).all()
assert (results_df["memory_mb_training"] > 0).all()
assert (results_df["memory_mb_inference"] > 0).all()
assert (results_df["parameter_count"] > 0).all()
assert (results_df["flops_estimated"] > 0).all()

print("✅ All validations passed")
print("✅ NB04 Cell 12 completed")

✅ All validations passed
✅ NB04 Cell 12 completed


In [16]:
# ============================================================
# NB04 Cell 13: ZIP + Summary
# ============================================================

summary = f"""
NB04 Distributed ML Pipeline

Datasets: {len(DATASETS)}
Partition methods: {len(PARTITION_METHODS)}
Aggregations: {len(AGGREGATIONS)}
Seeds: {len(SEEDS)}

Total runs: {len(results_df)}
"""

with open(META_DIR / "nb04_summary.txt", "w") as f:
    f.write(summary)

zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    root_dir=str(OUTPUT_ROOT)
)

print("\n" + "=" * 80)
print("✅ NB04 COMPLETED SUCCESSFULLY")
print("=" * 80)
print(f"Results rows      : {len(results_df)}")
print(f"Best F1           : {results_df['f1'].max():.4f}")
print(f"Best ROC-AUC      : {results_df['roc_auc'].max():.4f}")
print(f"ZIP               : {zip_path}")
print("=" * 80)


✅ NB04 COMPLETED SUCCESSFULLY
Results rows      : 120
Best F1           : 0.9199
Best ROC-AUC      : 0.9978
ZIP               : /kaggle/working/NB-4-AGAIN-outputs.zip
